[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Schemas and Validation


## What you will be able to do

Write down, once, the shape a response must have, as a JSON Schema or as a Pydantic model, and check
every response against it, so that a changed response stops the program with an error for each
member that changed.


## The idea

### The problem

The **JSON in a Response** notebook checked the network document by hand: `get` for the elevation
one station lacks, `is None` for a status that can be `null`, `fromisoformat` for every date. Those
checks sit wherever the data is used, and each covers a gap someone noticed. Nothing checks the rest
of the document.

Documents change. The practice API has a preview of the next release of its network document, at
`/beta/network`, and it breaks the current shape in four places: an elevation sent as the text
`"12"`, a flag sent as the word `"yes"`, a date written `30/06/2025`, and a station whose `id` has
been renamed. Code written for today's document meets each change somewhere different. The renamed
`id` raises `KeyError` wherever a station's id is read, and the date raises `ValueError` wherever it
is parsed. `"12"` prints as `12 m` and fails only when something adds to it or compares it, and
`"yes"` raises nothing at all: every non-empty string counts as true in an `if`, `"no"` included.

A schema puts those checks in one place and runs them at one moment. The response is checked
against the whole shape it promised as it arrives, and every broken promise is reported at once,
with the path to where it is.

### What a schema is

> A **schema** describes the shape data must have: the members it must contain, the type of each,
> and rules for their values, such as a range or a format. **Validation** checks data against a
> schema and reports each place the data breaks it. **JSON Schema** writes a schema as JSON, so any
> language can use it. **Pydantic** writes a schema as a Python class, and turns data that passes
> into objects of that class.

### Why it works that way

- **Check at the boundary.** Data is checked once, where it enters the program, so the code after
  that point can rely on its shape instead of guarding every lookup.
- **Report every error.** A schema check lists each place the data breaks the schema, with a path to
  it, where a hand-written check stops at the first problem.
- **A new member breaks nothing.** APIs add members as they grow, so JSON Schema and Pydantic both
  accept members a schema does not mention, unless told otherwise. A member that is missing, or has
  changed type, is what they report.
- **Missing and `null` are declared separately.** A schema says whether a member may be absent, and
  separately whether it may be `null`: the two cases the **JSON in a Response** notebook told apart
  by hand.
- **JSON Schema describes, and Pydantic converts.** `jsonschema` answers whether data matches.
  Pydantic also builds objects, turning `"2025-10-14"` into a `date`, and by default it converts
  values that are close enough, such as the text `"12"` into the number `12`. Strict mode stops the
  conversion.
- **Documentation can be a schema.** An OpenAPI document describes its bodies in JSON Schema, so the
  practice API's documentation, which the **Exploring an API** notebook read, can check the practice
  API's own responses.

### Where you will meet this

OpenAPI documents describe request and response bodies with JSON Schema, and editors such as VS Code
check configuration files against JSON Schema as you type. Pydantic is the validation library that
FastAPI is built on: the **Validating Requests** notebook uses models like the ones here to check
what clients send, and the **A Real Client** notebook checks every response its client receives.

### Installing them

Colab has `jsonschema` and `pydantic` already. On your own computer, `pip install jsonschema
pydantic` adds both, inside a virtual environment as the **Environments and pip** notebook explains.

### What this notebook covers

- A JSON Schema for the network document, and every error in a changed document, with its path
- The practice API's own OpenAPI schemas, checking its stations
- Pydantic models, with members that may be missing or `null`, and dates turned into dates
- What Pydantic converts, and strict mode, which stops it
- A real API's response, declared as far as a program uses it
- A JSON Schema generated from a model
- A client that refuses a changed response, with an error for each change
- Five errors, from a member Pydantic still requires to a model indexed like a dictionary

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests
from pydantic import BaseModel, ValidationError

class Station(BaseModel):
    id: str
    name: str
    latitude: float
    longitude: float

print(Station.model_validate(requests.get("http://127.0.0.1:8765/stations/tromso", timeout=10).json()))

try:
    Station.model_validate({"id": "narvik", "name": "Narvik", "latitude": "north"})
except ValidationError as error:
    for problem in error.errors():
        print(problem["loc"], problem["msg"])
```

```
id='tromso' name='Tromso' latitude=69.65 longitude=18.96
('latitude',) Input should be a valid number, unable to parse string as a number
('longitude',) Field required
```

A class that says what a station must be, a response that matches it, and a made-up station that
does not, with an error for each broken promise.


## Setup

Nine imports, the last of them the practice API.

- `requests` sends every request in this notebook
- `Draft202012Validator` and `FormatChecker`, from `jsonschema`, check data against a JSON Schema,
  formats included
- `BaseModel`, `Field` and `ValidationError`, from `pydantic`, declare a schema as a class, set rules
  on a member, and report what breaks them
- `date` and `datetime` are the types that dates in a model become
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`.
  `open_meteo()` returns Open-Meteo's address, or the address of the practice API's recording of
  it when Open-Meteo is not answering

If Open-Meteo stops answering while you work through the notebook, run this cell again: it checks
again, and the Open-Meteo cells switch to the recording.


In [1]:
import importlib
import sys
import urllib.request
from datetime import date, datetime
from pathlib import Path

import requests
from jsonschema import Draft202012Validator, FormatChecker
from pydantic import BaseModel, Field, ValidationError

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
OPEN_METEO = practice_api.open_meteo()
print("The practice API is running at", BASE)
print("Open-Meteo's archive is at", OPEN_METEO)


The practice API is running at http://127.0.0.1:8765
Open-Meteo's archive is at https://archive-api.open-meteo.com/v1/archive


## Worked examples

### A JSON Schema for the network document

A JSON Schema is a dictionary of keywords. `type` names the kind of value, `required` lists the
members that must be present, `properties` gives a schema for each member, and `minimum`,
`maximum` and `format` set rules on values. Here is the network document's shape, built from the
inside out, a schema for each kind of object:


In [2]:
location_schema = {"type": "object", "required": ["latitude", "longitude"],
                   "properties": {"latitude": {"type": "number", "minimum": -90, "maximum": 90},
                                  "longitude": {"type": "number", "minimum": -180, "maximum": 180},
                                  "elevation_m": {"type": "integer"}}}

instrument_schema = {"type": "object", "required": ["kind", "installed", "last_calibrated"],
                     "properties": {"kind": {"type": "string"},
                                    "installed": {"type": "string", "format": "date"},
                                    "last_calibrated": {"type": ["string", "null"], "format": "date"}}}

status_schema = {"type": ["object", "null"], "required": ["active", "issues"],
                 "properties": {"active": {"type": "boolean"}, "issues": {"type": "array"}}}

station_schema = {"type": "object", "required": ["id", "name", "location", "instruments", "status"],
                  "properties": {"id": {"type": "string"}, "name": {"type": "string"},
                                 "location": location_schema,
                                 "instruments": {"type": "array", "items": instrument_schema},
                                 "status": status_schema}}

network_schema = {"type": "object", "required": ["name", "updated", "stations"],
                  "properties": {"name": {"type": "string"}, "updated": {"type": "string"},
                                 "stations": {"type": "array", "items": station_schema}}}


The gaps the **JSON in a Response** notebook found are declared here. `elevation_m` is described
but not required, so a location may leave it out. `last_calibrated` is required, and its type is
`["string", "null"]`, so it must be present and may be `null`. `status` may be an object or `null`,
and when it is an object, it must have `active` and `issues`.

`Draft202012Validator` checks data against a schema written in the 2020-12 version of JSON Schema,
the one OpenAPI documents use, and `FormatChecker` makes it check `format` too:


In [3]:
validator = Draft202012Validator(network_schema, format_checker=FormatChecker())
network = requests.get(f"{BASE}/network", timeout=10).json()

print("errors:", list(validator.iter_errors(network)))


errors: []


`iter_errors` yields every error it finds, and the current document has none.

### Every broken promise, with its path

Here is the preview of the next release, checked against the same schema:


In [4]:
beta = requests.get(f"{BASE}/beta/network", timeout=10).json()

for error in validator.iter_errors(beta):
    print(f"{error.json_path:<46} {error.message}")


$.stations[0].location.elevation_m             '12' is not of type 'integer'
$.stations[1].status.active                    'yes' is not of type 'boolean'
$.stations[2].instruments[0].last_calibrated   '30/06/2025' is not a 'date'
$.stations[3]                                  'id' is a required property


Four errors, each with its path written the way the **JSON in a Response** notebook wrote paths,
and each naming the promise that broke: a type, a format, a required member. The renamed `id`
appears as a required member missing from `stations[3]`, because the schema checks what should be
there, not what the member is now called. `validate`, from the same library, raises the single most
relevant error instead, and `iter_errors` is the one to use when a report should list them all.

Two changes are not in the list: the new top-level member `version`, and `station_id`, the new name
of Tromso's `id`. A schema allows members it does not mention, which lets an API add members
without breaking its clients. Adding `"additionalProperties": False` to a schema forbids them.

### The practice API's own schemas

The **Exploring an API** notebook read the practice API's OpenAPI document, and found a `Station`
schema under `components`. It is JSON Schema, so the same validator can use it:


In [5]:
doc = requests.get(f"{BASE}/openapi.json", timeout=10).json()
documented_station = doc["components"]["schemas"]["Station"]

oslo = requests.get(f"{BASE}/stations/oslo", timeout=10).json()
print("Station errors:", list(Draft202012Validator(documented_station).iter_errors(oslo)))


Station errors: []


The response for `/stations` is documented as an array of `$ref`s. A `$ref` beginning with `#` is a
path from the root of the document it sits in, so the schema has to travel with the `components` it
points into. Without them there is nothing at `#/components/schemas/StationSummary`, and
`jsonschema` raises an error naming the pointer it could not follow. With them, the list checks:


In [6]:
response_schema = doc["paths"]["/stations"]["get"]["responses"]["200"]["content"]["application/json"]["schema"]
list_schema = {**response_schema, "components": doc["components"]}

stations = requests.get(f"{BASE}/stations", timeout=10).json()
print(response_schema)
print("list errors:", list(Draft202012Validator(list_schema).iter_errors(stations)))


{'type': 'array', 'items': {'$ref': '#/components/schemas/StationSummary'}}
list errors: []


Both responses match their documentation, the check the **Exploring an API** notebook made for
status codes, extended to bodies.

### A Pydantic model for the network

Pydantic writes a schema as a class. Each member is a class attribute with a type, and `Field` adds
rules. A member whose type is another model nests that model's schema inside:


In [7]:
class Location(BaseModel):
    latitude: float = Field(ge=-90, le=90)
    longitude: float = Field(ge=-180, le=180)
    elevation_m: int | None = None


class Instrument(BaseModel):
    kind: str
    installed: date
    last_calibrated: date | None


class Issue(BaseModel):
    since: date
    summary: str


class Status(BaseModel):
    active: bool
    issues: list[Issue]


class Station(BaseModel):
    id: str
    name: str
    location: Location
    instruments: list[Instrument]
    status: Status | None


class Network(BaseModel):
    name: str
    updated: datetime
    stations: list[Station]


The two gaps are declared the way the JSON Schema declared them. `elevation_m: int | None = None`
has a default, so it may be missing, and `None` is what a missing elevation becomes.
`last_calibrated: date | None` has no default, so it must be present, and may be `null`.
`model_validate` checks a dictionary against the model and returns an object of the class:


In [8]:
checked = Network.model_validate(network)

print(type(checked).__name__, checked.updated.date())
print(checked.stations[2].location)
print(checked.stations[1].instruments[1])
print(checked.stations[3].status)


Network 2026-03-01
latitude=78.22 longitude=15.65 elevation_m=None
kind='rain gauge' installed=datetime.date(2016, 3, 15) last_calibrated=None
None


Every level is an object now, reached with attributes instead of keys. The dates are `date` objects,
parsed from their text, and Svalbard's missing elevation and Tromso's `null` status are both `None`,
as declared. The code that uses a `Network` needs no `get`, no `dig`, and no `fromisoformat`.

### What Pydantic converts, and strict mode

Here is the preview of the next release, through the same model:


In [9]:
try:
    Network.model_validate(beta)
except ValidationError as error:
    for problem in error.errors():
        print(".".join(str(part) for part in problem["loc"]), "|", problem["msg"])


stations.2.instruments.0.last_calibrated | Input should be a valid date or datetime, invalid character in year
stations.3.id | Field required


Two errors, where the JSON Schema found four. Pydantic converted the other two without a word,
because by default it accepts a value it can turn into the declared type:


In [10]:
print(Location.model_validate(beta["stations"][0]["location"]))
print(Status.model_validate(beta["stations"][1]["status"]))


latitude=60.39 longitude=5.32 elevation_m=12
active=True issues=[]


The text `"12"` became the number `12`, and the word `"yes"` became `True`. That is convenient for
input a person typed, and it hides a changed response from an API. Strict mode converts nothing.
`model_validate_json` takes the response's JSON text rather than `response.json()`, and parses and
validates it in one step, which lets strict mode apply JSON's rules: a date in JSON can only be text,
so an ISO 8601 date is still accepted, and the text `"12"` for an integer is not:


In [11]:
response = requests.get(f"{BASE}/beta/network", timeout=10)

try:
    Network.model_validate_json(response.content, strict=True)
except ValidationError as error:
    for problem in error.errors():
        print(".".join(str(part) for part in problem["loc"]), "|", problem["msg"])


stations.0.location.elevation_m | Input should be a valid integer
stations.1.status.active | Input should be a valid boolean
stations.2.instruments.0.last_calibrated | Input should be a valid date in the format YYYY-MM-DD, invalid character in year
stations.3.id | Field required


All four changes, each with its location, which Pydantic writes as a tuple of names and positions,
joined here with dots.

### A real API's response, declared as far as the program uses it

A model for a third-party API does not have to describe the whole response. Declare the members the
program uses, and the rest are ignored. Here is Open-Meteo's response for Tromso, from the request the
**Query Parameters** notebook sent:


In [12]:
class Daily(BaseModel):
    time: list[date]
    temperature_2m_max: list[float]
    temperature_2m_min: list[float]
    precipitation_sum: list[float]


class Archive(BaseModel):
    latitude: float
    longitude: float
    daily_units: dict[str, str]
    daily: Daily


response = requests.get(OPEN_METEO, timeout=30, params={
    "latitude": 69.65, "longitude": 18.96, "start_date": "2025-01-15", "end_date": "2025-01-17",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum", "models": "era5"})
archive = Archive.model_validate_json(response.content, strict=True)

print(archive.latitude, archive.longitude)
print(archive.daily.time)
print(archive.daily.precipitation_sum, archive.daily_units["precipitation_sum"])


69.75 19.0
[datetime.date(2025, 1, 15), datetime.date(2025, 1, 16), datetime.date(2025, 1, 17)]
[23.7, 21.1, 18.8] mm


`list[date]` turned every day into a `date`. The response carries more than the model declares, such
as `elevation` and `generationtime_ms`, and none of it was checked or kept. If Open-Meteo stopped
sending a member the program uses, or sent it as a different type, `model_validate_json` would
raise there and then, rather than the program failing later. The coordinates are the grid point
Open-Meteo used, near the ones asked for.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.

### A JSON Schema generated from a model

A model can write its schema out as JSON Schema, so a schema written as a class can still be shared
with code in other languages:


In [13]:
generated = Location.model_json_schema()

print(generated["required"])
print(generated["properties"]["latitude"])
print(generated["properties"]["elevation_m"])
print("network errors:", list(Draft202012Validator(Network.model_json_schema(), format_checker=FormatChecker()).iter_errors(network)))


['latitude', 'longitude']
{'maximum': 90, 'minimum': -90, 'title': 'Latitude', 'type': 'number'}
{'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'title': 'Elevation M'}
network errors: []


The rules from `Field` became `minimum` and `maximum`, and `int | None = None` became a type of
integer or `null`, left out of `required`. The generated schema for `Network` checks the network
document with `jsonschema`, and finds nothing wrong. FastAPI puts schemas generated this way into the
OpenAPI documents it publishes, which is where the **Your First API Server** notebook's documentation
comes from.

### A client that refuses a changed response

Everything in this notebook, in one client. `fetch_network` validates a network document strictly
the moment it arrives, and `report` prints each station's state with nothing to guard against,
because every gap is declared in the model. Pointed at the preview of the next release, the client
refuses it, with an error for each change:


In [14]:
def fetch_network(url):
    """The network document at url, checked strictly against Network as it arrives."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return Network.model_validate_json(response.content, strict=True)


def report(network):
    """A line for each station: elevation, state, and the instruments due for calibration."""
    updated = network.updated.date()
    for station in network.stations:
        elevation = station.location.elevation_m
        where = "elevation unknown" if elevation is None else f"{elevation} m"
        if station.status is None:
            state = "status not reported"
        elif station.status.active:
            state = "active"
        else:
            state = "inactive: " + "; ".join(issue.summary for issue in station.status.issues)
        due = [instrument.kind for instrument in station.instruments
               if instrument.last_calibrated is None or (updated - instrument.last_calibrated).days > 365]
        print(f"  {station.name}, {where}, {state}; due for calibration: {', '.join(due) or 'nothing'}")


for url in [f"{BASE}/network", f"{BASE}/beta/network"]:
    print(url.removeprefix(BASE))
    try:
        report(fetch_network(url))
    except ValidationError as error:
        print(f"  refused, with {error.error_count()} changes:")
        for problem in error.errors():
            print("   ", ".".join(str(part) for part in problem["loc"]), "|", problem["msg"])


/network
  Bergen, 12 m, active; due for calibration: nothing
  Oslo, 94 m, active; due for calibration: rain gauge, anemometer
  Svalbard, elevation unknown, inactive: rain gauge buried in snow; due for calibration: rain gauge
  Tromso, 100 m, status not reported; due for calibration: anemometer
/beta/network
  refused, with 4 changes:
    stations.0.location.elevation_m | Input should be a valid integer
    stations.1.status.active | Input should be a valid boolean
    stations.2.instruments.0.last_calibrated | Input should be a valid date in the format YYYY-MM-DD, invalid character in year
    stations.3.id | Field required


### Where each part came from

| In the client | What it relies on | The section that showed it |
|---|---|---|
| `Network.model_validate_json(response.content, strict=True)` | parsing and checking in one step, converting nothing | What Pydantic converts, and strict mode |
| `station.location.elevation_m` | a member that may be missing, declared with a default | A Pydantic model for the network |
| `station.status is None` | a member that may be `null`, declared as a `Status` or `None` | A Pydantic model for the network |
| `updated - instrument.last_calibrated` | dates that arrive as `date` objects | A Pydantic model for the network |
| `error.errors()`, and each error's `loc` | every broken promise, with the path to it | Every broken promise, with its path |

`report` is the calibration report from the **JSON in a Response** notebook, shortened to a line a
station, with every check for a gap replaced by the model's declaration of it.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/07-schemas-and-validation-solutions.ipynb).

**1.** Check `/stations/svalbard` against `documented_station`, the practice API's own `Station`
schema, and print how many errors there are.


In [15]:
# your code here


**2.** Check the made-up station `{"id": "bodo", "name": "Bodo", "latitude": 67.28}` against
`documented_station`, and print the path and message of every error.


In [16]:
# your code here


**3.** Make a copy of `station_schema` with `"additionalProperties": False` added, check Tromso's
station from `/beta/network` against it, and print every message.


In [17]:
# your code here


**4.** Write a Pydantic model `Summary` for an item of `/stations`, validate every item of that list
with it, and print the `name` attribute of each.


In [18]:
# your code here


**5.** Validate `/network` with `Network` in strict mode. Print the names of the stations whose
`status` is `None`, and the kind of every instrument last calibrated before 2025.


In [19]:
# your code here


**6.** Validation ignores members a model does not declare. Using `Station.model_fields`, print the
set of member names that the stations of `/beta/network` have and `Station` does not declare.


In [20]:
# your code here


## Common errors

### ValidationError: Field required, for a member that may be None


In [21]:
class LocationWithoutDefault(BaseModel):
    latitude: float
    longitude: float
    elevation_m: int | None


for item in network["stations"]:
    LocationWithoutDefault.model_validate(item["location"])


ValidationError: 1 validation error for LocationWithoutDefault
elevation_m
  Field required [type=missing, input_value={'latitude': 78.22, 'longitude': 15.65}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

`int | None` says what the value may be, `None` included, and nothing about whether the member may
be left out. With no default, Pydantic requires the member, so Svalbard's location, which has no
`elevation_m`, fails. A default is what makes a member optional:


In [22]:
class LocationWithDefault(BaseModel):
    latitude: float
    longitude: float
    elevation_m: int | None = None


print([LocationWithDefault.model_validate(item["location"]).elevation_m for item in network["stations"]])


[12, 94, None, 100]


### No error, and a date that is not a date: a format without a format checker


In [23]:
unchecked = Draft202012Validator(network_schema)

for error in unchecked.iter_errors(beta):
    print(f"{error.json_path:<46} {error.message}")


$.stations[0].location.elevation_m             '12' is not of type 'integer'
$.stations[1].status.active                    'yes' is not of type 'boolean'
$.stations[3]                                  'id' is a required property


Three errors, for four changes: `30/06/2025` passed. In JSON Schema, `format` is only a note unless
the validator is given a format checker, which is why every validator in the worked examples was
built with `FormatChecker()`. With one, the date is reported:


In [24]:
checking = Draft202012Validator(network_schema, format_checker=FormatChecker())

print([error.json_path for error in checking.iter_errors(beta) if error.validator == "format"])


['$.stations[2].instruments[0].last_calibrated']


### ValidationError: Input should be a valid date, in strict mode on response.json()


In [25]:
Instrument.model_validate(network["stations"][0]["instruments"][0], strict=True)


ValidationError: 2 validation errors for Instrument
installed
  Input should be a valid date [type=date_type, input_value='2018-06-01', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/date_type
last_calibrated
  Input should be a valid date [type=date_type, input_value='2025-10-14', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/date_type

`network` came from `response.json()`, which has already turned the body into Python values, where a
date is still a string. In strict mode, a `date` member accepts only a `date` object, so both dates
fail. Give strict mode the JSON text instead: in JSON a date can only be text, so
`model_validate_json` in strict mode accepts an ISO 8601 date and nothing else:


In [26]:
strictly = Network.model_validate_json(requests.get(f"{BASE}/network", timeout=10).content, strict=True)

print(strictly.stations[0].instruments[0])


kind='thermometer' installed=datetime.date(2018, 6, 1) last_calibrated=datetime.date(2025, 10, 14)


### ValidationError: Input should be a valid dictionary or instance of Network


In [27]:
Network.model_validate(requests.get(f"{BASE}/network", timeout=10).text)


ValidationError: 1 validation error for Network
  Input should be a valid dictionary or instance of Network [type=model_type, input_value='{"name": "Practice API s...1"}], "status": null}]}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type

`text` is the body as a string, and `model_validate` does not parse JSON: it takes Python values,
and for a model that means a dictionary. Parse the body first, or give the text to
`model_validate_json`, which parses and validates in one step:


In [28]:
response = requests.get(f"{BASE}/network", timeout=10)

print(Network.model_validate(response.json()).name)
print(Network.model_validate_json(response.text).name)


Practice API station network
Practice API station network


### TypeError: 'Network' object is not subscriptable


In [29]:
checked["stations"]


TypeError: 'Network' object is not subscriptable

A model is an object, not a dictionary: its members are attributes. Use `checked.stations`, and
`model_dump()` when a dictionary is what the code needs:


In [30]:
print(len(checked.stations))
print(list(checked.model_dump()))


4
['name', 'updated', 'stations']


## Recap

- A schema describes the shape data must have, and validating checks data against it, reporting each
  place the data breaks it.
- A JSON Schema is a dictionary of keywords such as `type`, `required`, `properties`, `minimum` and
  `format`, checked with `Draft202012Validator`, which checks formats only with a `FormatChecker`.
- `iter_errors` lists every error, each with a `json_path` to where it is and a `message`.
- An OpenAPI document's schemas are JSON Schema; send its `components` along for a `$ref` to resolve.
- A Pydantic model declares members as typed attributes: a default for a member that may be missing,
  and `| None` for one that may be `null`. Data that passes becomes objects, with dates as `date`.
- By default Pydantic converts close-enough values, so check a response with
  `model_validate_json(response.content, strict=True)`.


## What is next

The **Headers and Content Types** notebook. Every response here was JSON, and the code assumed it;
that notebook reads the headers that say what a body is, and sends the headers that ask a server for
the format a client wants.


---

&#8592; **Previous:** [JSON in a Response](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/06-json-in-a-response.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
